# Next-Gen Kashmiri VITS & Piper TTS Training Pipeline (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sameerahmad86232/cipher/blob/main/colab-kashmiri-vits-training.ipynb)

Train an end-to-end **VITS (Variational Inference with Adversarial Learning)** neural speech synthesis model on Kashmiri studio audio. Exports a lightweight, zero-latency **Piper ONNX** model that runs directly in web browsers and mobile apps at 40× real-time speed.

**Hardware Recommendation:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
# Step 1: Verify NVIDIA T4 GPU
!nvidia-smi

In [ ]:
# Step 2: Install Audio & Export Dependencies
!pip install -q soundfile librosa onnx onnxruntime huggingface_hub pandas tqdm

In [ ]:
# Step 3: Clone Cipher Repository
!git clone https://github.com/sameerahmad86232/cipher.git /content/cipher
%cd /content/cipher

In [ ]:
# Step 4: Download Kashmiri Studio Audio Dataset from Kaggle
import os
import getpass

# Enter your Kaggle credentials securely
if not os.getenv("KAGGLE_USERNAME"):
    os.environ["KAGGLE_USERNAME"] = input("Enter Kaggle Username (e.g. sameerahmadnajar): ").strip()
if not os.getenv("KAGGLE_KEY"):
    os.environ["KAGGLE_KEY"] = getpass.getpass("Enter Kaggle API Key / Token: ").strip()

!pip install -q kaggle
!mkdir -p /content/ktts_dataset
!kaggle datasets download -d gopikrishnan2005/kashmiri-text-to-speech -p /content/ktts_dataset --unzip
print("[*] Dataset downloaded and extracted successfully!")

In [ ]:
# Step 5: Launch VITS Training & ONNX Export
import os

os.environ["BATCH_SIZE"] = "16"
os.environ["NUM_EPOCHS"] = "100"
os.environ["OUTPUT_DIR"] = "/content/kashmiri-vits-export"
os.environ["CHECKPOINTS_DIR"] = "/content/vits-checkpoints"

# Make kaggle input path point to our extracted dataset
!mkdir -p /kaggle/input
!ln -sfn /content/ktts_dataset /kaggle/input/kashmiri-text-to-speech

!python3 /content/cipher/scripts/finetune-vits-kashmiri.py

In [ ]:
# Step 6: Test Synthesizer in Colab (Type any Kashmiri sentence!)
import onnxruntime as ort
import numpy as np
import soundfile as sf
import json
import IPython.display as ipd

# Load Piper ONNX Model
session = ort.InferenceSession("/content/kashmiri-vits-export/kashmiri_vits.onnx")
config = json.load(open("/content/kashmiri-vits-export/kashmiri_vits.json"))
char_to_id = config["phoneme_id_map"]

test_sentence = "کٲشُر زَبان چھِ اَکھ سؠٹھاہ خوٗبصوٗرَت زَبان۔"
seq = [char_to_id[ch] for ch in test_sentence if ch in char_to_id]
text_tensor = np.array([seq], dtype=np.int64)
len_tensor = np.array([len(seq)], dtype=np.int64)

outputs = session.run(None, {"text": text_tensor, "text_lengths": len_tensor})
audio_wav = outputs[0][0, 0]
sf.write("/content/sample_speech.wav", audio_wav, samplerate=22050)

print("Generated Kashmiri Speech:")
ipd.Audio("/content/sample_speech.wav")

In [ ]:
# Step 7: Push Model Directly to Hugging Face Hub
import os
import getpass
from huggingface_hub import HfApi

hf_token = os.getenv("HF_TOKEN") or getpass.getpass("Enter Hugging Face Token (write access): ").strip()
api = HfApi(token=hf_token)
repo_id = "Sameer0313/kashmiri-vits-tts"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="/content/kashmiri-vits-export",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload Next-Gen Kashmiri VITS and Piper ONNX TTS model"
)
print(f"[+] Model uploaded to Hugging Face: https://huggingface.co/{repo_id}")